In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import glob
import os

In [18]:
df = pd.read_csv("../anac_data/Results/all_fields_gc_photometry_corrected_errors_v17.csv")
print("Numbers of sources", len(df))
df.columns.tolist()

Numbers of sources 3799


['recno',
 'T17ID',
 'oldID',
 'RAJ2000',
 'DEJ2000',
 'Prob',
 'Rgc',
 'PA',
 'umag',
 'gmag',
 'rmag',
 'imag',
 'zmag',
 'e_umag',
 's_umag',
 'e_gmag',
 's_gmag',
 'e_rmag',
 's_rmag',
 'e_imag',
 's_imag',
 'e_zmag',
 's_zmag',
 'FLUX_F378_2',
 'FLUXERR_F378_2',
 'MAG_F378_2',
 'MAGERR_F378_2',
 'SNR_F378_2',
 'AP_CORR_F378_2',
 'FLUX_F378_3',
 'FLUXERR_F378_3',
 'MAG_F378_3',
 'MAGERR_F378_3',
 'SNR_F378_3',
 'AP_CORR_F378_3',
 'FLUX_F395_2',
 'FLUXERR_F395_2',
 'MAG_F395_2',
 'MAGERR_F395_2',
 'SNR_F395_2',
 'AP_CORR_F395_2',
 'FLUX_F395_3',
 'FLUXERR_F395_3',
 'MAG_F395_3',
 'MAGERR_F395_3',
 'SNR_F395_3',
 'AP_CORR_F395_3',
 'FLUX_F410_2',
 'FLUXERR_F410_2',
 'MAG_F410_2',
 'MAGERR_F410_2',
 'SNR_F410_2',
 'AP_CORR_F410_2',
 'FLUX_F410_3',
 'FLUXERR_F410_3',
 'MAG_F410_3',
 'MAGERR_F410_3',
 'SNR_F410_3',
 'AP_CORR_F410_3',
 'FLUX_F430_2',
 'FLUXERR_F430_2',
 'MAG_F430_2',
 'MAGERR_F430_2',
 'SNR_F430_2',
 'AP_CORR_F430_2',
 'FLUX_F430_3',
 'FLUXERR_F430_3',
 'MAG_F430_3',
 'M

In [19]:
# Análisis de distribución de errores por filtro
filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
apertures = ['2', '3']

In [20]:
print("=== DISTRIBUCIÓN DE ERRORES POR FILTRO (percentiles) ===")
for filter in filters:
    for ap in apertures:
        mag_err_col = f'MAGERR_{filter}_{ap}'
        if mag_err_col in df.columns:
            errors = df[mag_err_col].replace(99.0, np.nan).dropna()
            if len(errors) > 0:
                p25, p50, p75, p90 = np.percentile(errors, [25, 50, 75, 90])
                print(f"{filter}_{ap}: P50={p50:.3f}, P75={p75:.3f}, P90={p90:.3f}, N={len(errors)}")

=== DISTRIBUCIÓN DE ERRORES POR FILTRO (percentiles) ===
F378_2: P50=0.543, P75=0.543, P90=0.543, N=2509
F378_3: P50=0.543, P75=0.543, P90=0.543, N=2471
F395_2: P50=0.543, P75=0.543, P90=0.543, N=2455
F395_3: P50=0.543, P75=0.543, P90=0.543, N=2469
F410_2: P50=0.543, P75=0.543, P90=0.543, N=2711
F410_3: P50=0.543, P75=0.543, P90=0.543, N=2619
F430_2: P50=0.543, P75=0.543, P90=0.543, N=2789
F430_3: P50=0.543, P75=0.543, P90=0.543, N=2704
F515_2: P50=0.446, P75=0.543, P90=0.543, N=3110
F515_3: P50=0.460, P75=0.543, P90=0.543, N=3057
F660_2: P50=0.140, P75=0.263, P90=0.449, N=3312
F660_3: P50=0.153, P75=0.279, P90=0.489, N=3302
F861_2: P50=0.225, P75=0.373, P90=0.543, N=3293
F861_3: P50=0.250, P75=0.417, P90=0.543, N=3279


In [21]:
def clean_photometry_catalog(df, 
                           blue_error_thresh=0.5, 
                           red_error_thresh=0.3,
                           min_filters=4):
    """
    Limpia el catálogo fotométrico con criterios de calidad
    
    Parameters:
    - blue_error_thresh: Error máximo para filtros azules (F378, F395, F410)
    - red_error_thresh: Error máximo para filtros rojos (F515, F660, F861)  
    - min_filters: Mínimo número de filtros con mediciones válidas
    """
    
    # Crear copia
    clean_df = df.copy()
    
    # Definir filtros por tipo
    blue_filters = ['F378', 'F395', 'F410', 'F430']  # Incluyo F430 como transición
    red_filters = ['F515', 'F660', 'F861']
    
    # Contador de filtros válidos por fuente
    valid_filters_count = np.zeros(len(clean_df))
    
    # Aplicar criterios de error para cada filtro y apertura
    for filter in blue_filters + red_filters:
        for ap in ['2', '3']:
            mag_col = f'MAG_{filter}_{ap}'
            err_col = f'MAGERR_{filter}_{ap}'
            
            if mag_col in clean_df.columns and err_col in clean_df.columns:
                # Definir umbral según tipo de filtro
                if filter in blue_filters:
                    error_threshold = blue_error_thresh
                else:
                    error_threshold = red_error_thresh
                
                # Crear máscara de datos válidos
                valid_mask = (
                    (clean_df[mag_col] < 90) &  # Excluir valores de error (99.0)
                    (clean_df[err_col] < error_threshold) &
                    (clean_df[err_col] > 0) &  # Excluir errores cero o negativos
                    (clean_df[f'SNR_{filter}_{ap}'] > 2)  # SNR mínimo
                )
                
                # Contar filtros válidos
                valid_filters_count += valid_mask.astype(int)
                
                # Opcional: poner NaN en mediciones inválidas
                # clean_df.loc[~valid_mask, mag_col] = np.nan
                # clean_df.loc[~valid_mask, err_col] = np.nan
    
    # Filtrar fuentes con mínimo número de filtros válidos
    mask_min_filters = valid_filters_count >= min_filters
    clean_df = clean_df[mask_min_filters].copy()
    
    print(f"Fuentes originales: {len(df)}")
    print(f"Fuentes después de limpieza: {len(clean_df)}")
    print(f"Retención: {len(clean_df)/len(df)*100:.1f}%")
    print(f"Filtros válidos por fuente: min={valid_filters_count.min()}, max={valid_filters_count.max()}, mean={valid_filters_count.mean():.1f}")
    
    return clean_df

In [22]:
# Aplicar limpieza
clean_df = clean_photometry_catalog(df, 
                                  blue_error_thresh=0.5, 
                                  red_error_thresh=0.3,
                                  min_filters=4)

Fuentes originales: 3799
Fuentes después de limpieza: 1935
Retención: 50.9%
Filtros válidos por fuente: min=0.0, max=14.0, mean=4.2


## 3. Versión Más Estricta (Para Paper High-Impact)

In [23]:
def strict_cleaning(df):
    """Limpieza estricta para análisis de alta calidad"""
    
    # 1. Criterios de error más estrictos
    clean_df = clean_photometry_catalog(df, 
                                      blue_error_thresh=0.4,  # Más estricto en azul
                                      red_error_thresh=0.2,   # Más estricto en rojo
                                      min_filters=5)         # Mínimo 5 filtros
    
    # 2. Calidad adicional basada en coherencia interna
    coherence_threshold = 0.1  # Diferencia mediana entre aperturas
    
    if 'COHERENCE_MEDIAN_DIFF' in clean_df.columns:
        coherence_mask = clean_df['COHERENCE_MEDIAN_DIFF'].abs() < coherence_threshold
        clean_df = clean_df[coherence_mask]
        print(f"Después de coherencia aperturas: {len(clean_df)} fuentes")
    
    # 3. Filtrar por probabilidad de cúmulo globular (si disponible)
    if 'Prob' in clean_df.columns:
        prob_mask = clean_df['Prob'] > 0.7  # Alta probabilidad de GC
        clean_df = clean_df[prob_mask]
        print(f"Después de probabilidad GC: {len(clean_df)} fuentes")
    
    return clean_df


In [24]:
# Aplicar limpieza estricta
strict_df = strict_cleaning(df)

Fuentes originales: 3799
Fuentes después de limpieza: 836
Retención: 22.0%
Filtros válidos por fuente: min=0.0, max=14.0, mean=3.0
Después de coherencia aperturas: 347 fuentes
Después de probabilidad GC: 200 fuentes


## 4. Análisis de Sensibilidad a Criterios

In [25]:
def sensitivity_analysis(df):
    """Analiza cómo cambia la muestra con diferentes criterios"""
    
    criteria_combinations = [
        {'blue_err': 0.6, 'red_err': 0.4, 'min_filt': 3, 'label': 'Muy Liberal'},
        {'blue_err': 0.5, 'red_err': 0.3, 'min_filt': 4, 'label': 'Moderado'},
        {'blue_err': 0.4, 'red_err': 0.2, 'min_filt': 5, 'label': 'Estricto'},
        {'blue_err': 0.3, 'red_err': 0.15, 'min_filt': 6, 'label': 'Muy Estricto'}
    ]
    
    print("=== ANÁLISIS DE SENSIBILIDAD ===")
    results = []
    
    for criteria in criteria_combinations:
        temp_df = clean_photometry_catalog(df, 
                                         criteria['blue_err'],
                                         criteria['red_err'], 
                                         criteria['min_filt'])
        results.append({
            'Criterio': criteria['label'],
            'Fuentes': len(temp_df),
            'Retención': f"{len(temp_df)/len(df)*100:.1f}%",
            'Blue_Err': criteria['blue_err'],
            'Red_Err': criteria['red_err'],
            'Min_Filt': criteria['min_filt']
        })
    
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))
    
    return results_df

In [26]:
# Ejecutar análisis de sensibilidad
sensitivity_results = sensitivity_analysis(df)

=== ANÁLISIS DE SENSIBILIDAD ===
Fuentes originales: 3799
Fuentes después de limpieza: 2622
Retención: 69.0%
Filtros válidos por fuente: min=0.0, max=14.0, mean=5.0
Fuentes originales: 3799
Fuentes después de limpieza: 1935
Retención: 50.9%
Filtros válidos por fuente: min=0.0, max=14.0, mean=4.2
Fuentes originales: 3799
Fuentes después de limpieza: 836
Retención: 22.0%
Filtros válidos por fuente: min=0.0, max=14.0, mean=3.0
Fuentes originales: 3799
Fuentes después de limpieza: 473
Retención: 12.5%
Filtros válidos por fuente: min=0.0, max=14.0, mean=2.1
    Criterio  Fuentes Retención  Blue_Err  Red_Err  Min_Filt
 Muy Liberal     2622     69.0%       0.6     0.40         3
    Moderado     1935     50.9%       0.5     0.30         4
    Estricto      836     22.0%       0.4     0.20         5
Muy Estricto      473     12.5%       0.3     0.15         6
